In [11]:
from dotenv import load_dotenv
load_dotenv()

import os
print("Key loaded:", os.getenv("GROQ_API_KEY") is not None)

Key loaded: True


In [12]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [13]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ]
)

print(response.choices[0].message.content)

Hello!


In [14]:
import requests

resp = requests.get(
    "https://clinicaltrials.gov/api/v2/studies",
    params={
        "query.cond": "rheumatoid arthritis",
        "filter.overallStatus": "RECRUITING",
        "fields": "NCTId,BriefTitle,EligibilityCriteria,MinimumAge,MaximumAge,Sex,Condition,OverallStatus",
        "pageSize": 10,
    },
)

data = resp.json()
print(len(data["studies"]))
print(data["studies"][0])

10
{'protocolSection': {'identificationModule': {'nctId': 'NCT06906549', 'briefTitle': 'Evaluation of 200 mg of Rituximab Every 6 Months as Maintenance Treatment of Rituximab-treated Patients With Rheumatoid Arthritis'}, 'statusModule': {'overallStatus': 'RECRUITING'}, 'conditionsModule': {'conditions': ['Rheumatoid Arthritis (RA)', 'Rituximab (RTx)']}, 'eligibilityModule': {'eligibilityCriteria': "Inclusion Criteria:\n\n* Age ≥ 18 years\n* Diagnosis of rheumatoid arthritis (RA) according to EULAR/ACR 2010 classification criteria\n* DAS28 ≤ 5.1\n* Current maintenance treatment with Rituximab regardless of dose and/or duration of Rituximab treatment and with at least first cycle of Rituximab ended (2 initial infusions)\n* Last Rituximab infusion between 6 and 18 months prior to inclusion\n* Corticosteroids ≤10 mg/day within 4 weeks prior to inclusion\n* Affiliation to a social insurance system or beneficiary\n* Written informed consent to participate in the study, dated and signed befor

In [15]:
def parse_study(study):
    protocol = study["protocolSection"]
    return {
        "nct_id": protocol["identificationModule"].get("nctId"),
        "title": protocol["identificationModule"].get("briefTitle"),
        "status": protocol["statusModule"].get("overallStatus"),
        "conditions": protocol.get("conditionsModule", {}).get("conditions", []),
        "eligibility_criteria": protocol.get("eligibilityModule", {}).get("eligibilityCriteria", ""),
        "sex": protocol.get("eligibilityModule", {}).get("sex"),
        "min_age": protocol.get("eligibilityModule", {}).get("minimumAge"),
        "max_age": protocol.get("eligibilityModule", {}).get("maximumAge"),
    }

trials = [parse_study(s) for s in data["studies"]]
trials[0]

{'nct_id': 'NCT06906549',
 'title': 'Evaluation of 200 mg of Rituximab Every 6 Months as Maintenance Treatment of Rituximab-treated Patients With Rheumatoid Arthritis',
 'status': 'RECRUITING',
 'conditions': ['Rheumatoid Arthritis (RA)', 'Rituximab (RTx)'],
 'eligibility_criteria': "Inclusion Criteria:\n\n* Age ≥ 18 years\n* Diagnosis of rheumatoid arthritis (RA) according to EULAR/ACR 2010 classification criteria\n* DAS28 ≤ 5.1\n* Current maintenance treatment with Rituximab regardless of dose and/or duration of Rituximab treatment and with at least first cycle of Rituximab ended (2 initial infusions)\n* Last Rituximab infusion between 6 and 18 months prior to inclusion\n* Corticosteroids ≤10 mg/day within 4 weeks prior to inclusion\n* Affiliation to a social insurance system or beneficiary\n* Written informed consent to participate in the study, dated and signed before starting the trial\n* Effective method of birth control during the study\n\nExclusion Criteria:\n\n* Rheumatic auto

In [16]:
def fetch_trials(condition, page_size=100):
    resp = requests.get(
        "https://clinicaltrials.gov/api/v2/studies",
        params={
            "query.cond": condition,
            "filter.overallStatus": "RECRUITING",
            "fields": "NCTId,BriefTitle,EligibilityCriteria,MinimumAge,MaximumAge,Sex,Condition,OverallStatus",
            "pageSize": page_size,
        },
    )
    return resp.json()["studies"]

conditions = ["rheumatoid arthritis", "breast cancer"]

all_trials = []
for cond in conditions:
    studies = fetch_trials(cond)
    all_trials.extend([parse_study(s) for s in studies])

print(len(all_trials))

200


In [28]:
from minsearch import Index

index = Index(
    text_fields=["title", "eligibility_criteria", "conditions", "nct_id"],
    keyword_fields=["nct_id"]
)

# minsearch expects conditions as text, not a list — join it
for t in all_trials:
    t["conditions"] = " ".join(t["conditions"]) if t["conditions"] else ""

index.fit(all_trials)

In [32]:
test_id = all_trials[0]["nct_id"]
results = index.search(test_id, num_results=3)
print(f"Testing with {test_id}: {len(results)} result(s) found")
for r in results:
    print(r["nct_id"], "-", r["title"])

Testing with NCT06906549: 1 result(s) found
NCT06906549 - Evaluation of 200 mg of Rituximab Every 6 Months as Maintenance Treatment of Rituximab-treated Patients With Rheumatoid Arthritis


In [18]:
results = index.search("rheumatoid arthritis patients over 45", num_results=3)
for r in results:
    print(r["nct_id"], "-", r["title"])

NCT03192267 - Early Rheumatoid Arthritis Lung Disease Study
NCT06503237 - Safety and Efficacy of SWK002 in Patients With D2T-Rheumatoid Arthritis
NCT07037576 - Efficacy and Safety of Prepectoral Prosthesis Immediate One-Stage Breast Reconstruction Versus Two-Stage Expander/Prosthesis Reconstruction in Postoperative Adjuvant Radiotherapy Breast Cancer Patients: A Prospective, Single-Center, Cohort Study


In [19]:
def build_prompt(query, results):
    context = "\n\n".join(
        f"NCT ID: {r['nct_id']}\nTitle: {r['title']}\nEligibility: {r['eligibility_criteria']}"
        for r in results
    )
    return f"""Answer the question based only on the clinical trial information below.
If the information isn't sufficient to answer, say so.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

def rag(query):
    results = index.search(query, num_results=3)
    prompt = build_prompt(query, results)
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(rag("What trials exist for rheumatoid arthritis patients over 45?"))

Based on the three study entries you provided, **two trials are recruiting (or could potentially recruit) rheumatoid‑arthritis (RA) patients who are older than 45 years**:

| NCT ID | Title | Age eligibility (covers > 45 y) | Key RA‑specific inclusion points |
|--------|-------|--------------------------------|-----------------------------------|
| **NCT03192267** | **Early Rheumatoid Arthritis Lung Disease Study** | 19 – 90 years (so includes anyone > 45) | • Diagnosis of RA per 2010 ACR criteria within the past 2 years.<br>• Must be able to give informed consent. |
| **NCT06503237** | **Safety and Efficacy of SWK002 in Patients With D2T‑Rheumatoid Arthritis** | ≥ 18 years (so includes anyone > 45) | • Adult patients who meet 2010 ACR/EULAR RA criteria and have **difficult‑to‑treat (D2T) RA** (failure of ≥ 2 bDMARDs/tsDMARDs, DAS28‑ESR > 3.2 or CDAI > 10, etc.).<br>• Must be on stable csDMARD therapy as defined in the protocol. |

### Why the other listed trial does **not** apply
- **

In [20]:
import re

def parse_age(age_str):
    if not age_str:
        return None
    match = re.search(r"\d+", age_str)
    return int(match.group()) if match else None

# quick test
print(parse_age("18 Years"), parse_age(None), parse_age("N/A"))

18 None None


In [21]:
def check_eligibility(nct_id, patient_age):
    trial = next((t for t in all_trials if t["nct_id"] == nct_id), None)
    if not trial:
        return {"error": "trial not found"}
    
    min_age = parse_age(trial["min_age"])
    max_age = parse_age(trial["max_age"])
    
    eligible = True
    reasons = []
    if min_age is not None and patient_age < min_age:
        eligible = False
        reasons.append(f"patient age {patient_age} is below minimum age {min_age}")
    if max_age is not None and patient_age > max_age:
        eligible = False
        reasons.append(f"patient age {patient_age} is above maximum age {max_age}")
    
    return {
        "nct_id": nct_id,
        "eligible_by_age": eligible,
        "min_age": min_age,
        "max_age": max_age,
        "reasons": reasons,
    }

# quick test — use a real nct_id from your data
print(check_eligibility(all_trials[0]["nct_id"], 46))

{'nct_id': 'NCT06906549', 'eligible_by_age': True, 'min_age': 18, 'max_age': None, 'reasons': []}


In [22]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_eligibility",
            "description": "Check if a patient of a given age is eligible for a specific clinical trial based on its age requirements.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nct_id": {"type": "string", "description": "The NCT ID of the trial"},
                    "patient_age": {"type": "integer", "description": "The patient's age in years"},
                },
                "required": ["nct_id", "patient_age"],
            },
        },
    }
]

In [23]:
def build_agentic_prompt(query, results):
    context = "\n\n".join(
        f"NCT ID: {r['nct_id']}\nTitle: {r['title']}\nEligibility: {r['eligibility_criteria']}"
        for r in results
    )
    return f"""You are a clinical trials assistant. You have access to a tool called
check_eligibility that checks whether a patient of a given age meets a trial's age requirements.

If the question mentions a specific NCT ID and a patient age, use the tool to check eligibility —
don't try to reason about ages yourself.

Otherwise, answer using the trial information below.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

In [24]:
def rag_agentic(query):
    nct_match = re.search(r"NCT\d{8}", query)
    
    if nct_match:
        # specific trial mentioned — skip retrieval, go straight to tool-enabled call
        messages = [{"role": "user", "content": query}]
    else:
        # general question — use retrieval as before
        results = index.search(query, num_results=3)
        prompt = build_agentic_prompt(query, results)
        messages = [{"role": "user", "content": prompt}]
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
    )
    
    msg = response.choices[0].message
    
    if msg.tool_calls:
        messages.append(msg)
        for call in msg.tool_calls:
            import json
            args = json.loads(call.function.arguments)
            result = check_eligibility(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
        final = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
        )
        return final.choices[0].message.content
    
    return msg.content

print(rag_agentic("Is a 70-year-old patient eligible for trial NCT07045896?"))

I’m sorry, but I couldn’t locate a trial with the identifier **NCT07045896** in the database. It’s possible that the number is mistyped or the trial isn’t listed in the source I’m accessing.

If you have an alternative NCT number (or any other details about the study, such as its title, sponsor, or therapeutic area), feel free to share it and I can look up the eligibility criteria for you.
